In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

spark.sql("USE plworkforce_catalog.002_silver")

# Read Bronze payroll
df = spark.table("plworkforce_catalog.001_bronze.payroll")

# Read Bronze employee to get base_salary
df_employee = spark.table("plworkforce_catalog.001_bronze.employee").select(
    "employee_id",
    F.col("base_salary").cast(DecimalType(18,2)).alias("base_salary")
)

# Join payroll with employee to get base_salary
df_joined = df.join(df_employee, "employee_id", "left")

# Transform
df_clean = (df_joined

    # Convert dates
    .withColumn("pay_period_start", F.to_timestamp("pay_period_start", "dd-MM-yyyy HH:mm"))
    .withColumn("pay_period_end", F.to_timestamp("pay_period_end", "dd-MM-yyyy HH:mm"))
    .withColumn("pay_date", F.to_timestamp("pay_date", "dd-MM-yyyy HH:mm"))

    # Cast numeric columns
    .withColumn("bonus", F.col("bonus").cast(DecimalType(18,2)))
    .withColumn("overtime_pay", F.col("overtime_pay").cast(DecimalType(18,2)))
    .withColumn("commission", F.col("commission").cast(DecimalType(18,2)))
    .withColumn("allowances", F.col("allowances").cast(DecimalType(18,2)))

    .withColumn("tax_deduction", F.col("tax_deduction").cast(DecimalType(18,2)))
    .withColumn("social_security", F.col("social_security").cast(DecimalType(18,2)))
    .withColumn("health_insurance", F.col("health_insurance").cast(DecimalType(18,2)))
    .withColumn("retirement_contribution", F.col("retirement_contribution").cast(DecimalType(18,2)))
    .withColumn("other_deductions", F.col("other_deductions").cast(DecimalType(18,2)))

    # Drop both gross_salary and net_salary from bronze data
    .drop("gross_salary", "net_salary")

    # Calculate gross_salary = base_salary + all variable pay components
    .withColumn("gross_salary",
        F.col("base_salary") +
        F.col("bonus") +
        F.col("overtime_pay") +
        F.col("commission") +
        F.col("allowances")
    )

    # Calculate net_salary using the formula:
    # base_salary + bonus + overtime_pay + commission + allowances - (all deductions)
    .withColumn("net_salary",
        F.col("base_salary") +
        F.col("bonus") +
        F.col("overtime_pay") +
        F.col("commission") +
        F.col("allowances") -
        (
            F.col("tax_deduction") +
            F.col("social_security") +
            F.col("health_insurance") +
            F.col("retirement_contribution") +
            F.col("other_deductions")
        )
    )

    # Standardize text
    .withColumn("status", F.upper(F.col("status")))
    .withColumn("payment_method", F.initcap(F.col("payment_method")))

    # Derived columns
    .withColumn("year", F.year("pay_date"))
    .withColumn("month", F.month("pay_date"))
    .withColumn("year_month", F.date_format("pay_date", "yyyy-MM"))

    # Data quality
    .dropDuplicates(["payroll_id"])
    .filter(F.col("employee_id").isNotNull())
    
    # Drop base_salary column (only needed for calculation)
    .drop("base_salary")
)

# Write to Silver with schema overwrite
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("payroll")
)

print("✓ Silver payroll created with recalculated gross_salary and net_salary")
print("  gross_salary = base_salary + bonus + overtime_pay + commission + allowances")
print("  net_salary = gross_salary - (tax_deduction + social_security + health_insurance + retirement_contribution + other_deductions)")

display(spark.read.table("plworkforce_catalog.002_silver.payroll"))